# exp09: Konteks Temporal Kausal

Menguji apakah riwayat fitur dari window sebelumnya membantu XGBoost dan Logistic Regression. Bandingkan clean tanpa konteks dengan tambahan rolling tiga atau lima window.

Target memakai kesulitan tugas sebagai proxy cognitive load: easy=Low, medium=Medium, hard=High.

### Akses Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Environment

Menyiapkan library dari requirements-final.txt dan memakai fungsi bersama shared.experiments untuk menjaga konsistensi preprocessing dan evaluasi.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib

ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE').resolve()
MODELS_DIR = ROOT / 'models'
manifest_path = MODELS_DIR / 'FINAL_MANIFEST.json'
if not manifest_path.is_file():
    raise RuntimeError('Upload FINAL_MANIFEST.json together with shared and requirements-final.txt')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for relative, expected in manifest['files'].items():
    if relative.startswith('shared/') or relative == 'requirements-final.txt':
        target = MODELS_DIR / relative
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Incomplete or outdated upload: {relative}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(MODELS_DIR / 'requirements-final.txt')])
import importlib.metadata
for module_name, package_name in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('xgboost', 'xgboost'), ('matplotlib', 'matplotlib'), ('joblib', 'joblib'), ('imblearn', 'imbalanced-learn'), ('catboost', 'catboost'), ('threadpoolctl', 'threadpoolctl')]:
    loaded = sys.modules.get(module_name)
    if loaded is not None and getattr(loaded, '__version__', None) != importlib.metadata.version(package_name):
        raise RuntimeError(f'Restart runtime before continuing: {module_name} was loaded before installation')


### Konfigurasi

Dataset utama adalah good_acceptable.csv. Mode full menjalankan evaluasi lengkap; seed utama 42. RESUME_DIR dapat diisi direktori run yang terputus. Dataset dan kode harus cocok dengan checkpoint.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

sys.path.insert(0, str(MODELS_DIR))
from shared.experiments.runner import Config, run
from shared.experiments.design import registry
from shared.data import load_dataset, describe_data, make_folds

EXPERIMENT = 'exp09'
MODE = 'full'
SEED = 42
RESUME_DIR = None
DATASET_PATH = ROOT / 'Dataset' / 'good_acceptable.csv'
OUTPUT_ROOT = MODELS_DIR / 'outputs'
config = Config(DATASET_PATH, OUTPUT_ROOT, EXPERIMENT, MODE, SEED)

### Dataset dan Kandidat

Tambahan temporal hanya untuk RMSSD, SDNN, rasio baseline RMSSD, blink rate, dan mean EAR kiri/kanan. Setiap varian mempertahankan window uji yang sama. Notebook memerlukan timestamp, peserta/sesi, puzzle_id, dan window_phase_mix.

Validasi schema, distribusi kelas per peserta, dan manifest outer leave-one-participant-out dilakukan sebelum training. Identitas peserta/soal bukan input model.

In [ ]:
data = load_dataset(DATASET_PATH)
print(describe_data(data))
display(pd.crosstab(data.participant_id, data.difficulty_level))
folds = make_folds(data)
specs = registry(EXPERIMENT, MODE, SEED)
display(pd.DataFrame([{'model': s['id'], 'features': s['variant'], 'scaler': s['scaler'], 'candidates': len(s['candidates']), 'inner_search': s['tune']} for s in specs]))
print('Outer folds:', len(folds))

### Preprocessing dan Training

Urutkan data dalam peserta/sesi. Reset riwayat saat soal berubah, window memotong beberapa soal, waktu tidak meningkat, atau gap melebihi 7,5 detik. Window campuran tidak mewariskan riwayat ke window berikutnya. Rolling mean memakai current/past dengan minimum satu window; selisih pertama bernilai nol. Fit imputasi dan scaler pada training, lalu latih model referensi tetap tanpa tuning.

Checkpoint per unit menyimpan hasil lengkap dengan checksum. Progres mencakup outer fold, kandidat, inner fold, dan waktu proses.

In [ ]:
run_dir = run(config, resume_dir=RESUME_DIR)

### Evaluasi

Bandingkan history3/history5 dengan history0 dalam model yang sama. Tidak ada data masa depan, centered rolling, atau penghapusan window awal. Tiga window 25 detik dengan langkah 5 detik mencakup rentang mentah 35 detik. Perubahan konteks ini tidak menghitung ulang HRV.

Metrik utama adalah macro-F1 gabungan outer test. Periksa accuracy, balanced accuracy, precision/recall/F1 tiap kelas, dan hasil per peserta. Kelas tanpa support dilaporkan kosong. Gunakan full evaluation untuk laporan penelitian.

In [ ]:
summary = json.loads((run_dir / 'summary.json').read_text())
print(summary)
if summary['status'] != 'complete':
    raise RuntimeError('Incomplete evaluation')
if MODE == 'smoke':
    print('Smoke test only. Change MODE to full and run again for research evaluation.')
display(pd.read_csv(run_dir / 'metrics.csv'))
classes = pd.read_csv(run_dir / 'class_metrics.csv')
display(classes[classes.scope == 'pooled'])
display(pd.read_csv(run_dir / 'participant_metrics.csv'))
display(pd.read_csv(run_dir / 'search_results.csv'))
print('Saved results:', run_dir)

### Confusion Matrix

Baris menunjukkan kelas sebenarnya, kolom menunjukkan prediksi. Proporsi per baris pada diagonal menunjukkan recall kelas; periksa pertukaran Low, Medium, dan High.

In [ ]:
for name in pd.read_csv(run_dir / 'metrics.csv').model:
    display(Image(filename=str(run_dir / 'figures' / f'{name}_normalized.png')))